# PokouAI — Fine-tune Gemma 4 on Cocoa Diseases

**Goal**: QLoRA fine-tune Gemma 4 (multimodal) on ~50k augmented cocoa pod images across 6 classes and 4 languages, export GGUF Q4_K_M for on-device inference.

**Target device**: entry-level Android, 2–3 GB RAM → **E2B is the primary variant** (~1.5 GB quantized, fits with OS + camera headroom). E4B is trained as an optional 'premium' variant for phones with ≥4 GB RAM.

**Platform**: Kaggle (T4 x2, 30 GB VRAM combined).

**Stack**: Unsloth (2× faster, ~60% less VRAM) + PEFT + TRL.

## Why Unsloth
Unsloth rewrites the Gemma forward/backward pass in Triton, enabling QLoRA fine-tuning on a single free-tier T4. Without it, QLoRA + gradient checkpointing still OOMs at sequence length 2048 with images. Unsloth makes E4B run at ~12 GB peak (E2B at ~7 GB).

## Licensing
Gemma 4 weights are distributed publicly under the Gemma Terms of Use. The direct `google/gemma-4-*-it` repos load without gated access — no click-through required. The notebook tries the direct HF repo first and falls back to Unsloth's pre-quantized 4-bit repo only if that fails.

## 1 — Environment setup

In [ ]:
# 1. New cell — uninstall + reinstall fresh
!pip uninstall -y pillow
!pip install --no-cache-dir "pillow==11.3.0"

In [ ]:
# Pillow ABI fix for Kaggle: pip pulls Pillow 12.2 but the compiled _imaging.so
# on disk is 11.3 → ImportError. Pin Pillow to 11.3.0 *after* unsloth installs
# its deps (which would otherwise pull Pillow 12 back in). Must be the LAST
# install command before any `from PIL import Image` runs.
!pip install --force-reinstall --no-cache-dir --no-deps "pillow==11.3.0"
print('✓ pillow pinned')

In [ ]:
!pip install -qU "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qU transformers accelerate peft trl datasets bitsandbytes pillow

In [ ]:
!pip install --force-reinstall --no-deps "pillow<12.0"

In [ ]:
import os, json, torch
from pathlib import Path
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

print('CUDA:', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}] {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

## 2 — Load Gemma 4 (E2B primary, E4B optional)

`VARIANT` controls which size to train. Defaults to `e2b` for on-device deployment.

Strategy per variant:
1. **Try direct HF repo** (`google/gemma-4-{variant}-it`) — public, no license click-through.
2. **Fall back** to Unsloth's pre-quantized 4-bit repo if the direct load errors (network, version mismatch, etc.).

In [ ]:
from datasets import load_dataset

# The Kaggle dataset is mounted at /kaggle/input/datasets/<owner>/<slug>/
# Adjust DATA_ROOT if your slug differs.
DATA_ROOT = Path('/kaggle/input/datasets/yaokouadio/pokou-ai-cocoa-training-data')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
VAL_JSONL = DATA_ROOT / 'val.jsonl'

print(f'loading train: {TRAIN_JSONL}')
train_ds = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
print(f'loading val:   {VAL_JSONL}')
val_ds = load_dataset('json', data_files=str(VAL_JSONL), split='train')

print(f'\nfull dataset:    train={len(train_ds):,}   val={len(val_ds):,}')

# --- subsample so we fit in Kaggle's 12h kernel + ~20 GB working disk ---
# 200k+ examples blow the disk via .map cache. 5k/500 trains in ~30 min on
# T4 and is plenty to validate the pipeline end-to-end. Bigger run after.
TRAIN_N = 5_000
VAL_N = 500
train_ds = train_ds.shuffle(seed=42).select(range(min(TRAIN_N, len(train_ds))))
val_ds = val_ds.shuffle(seed=42).select(range(min(VAL_N, len(val_ds))))
print(f'subsampled:      train={len(train_ds):,}   val={len(val_ds):,}')
print(f'first example keys: {list(train_ds[0].keys())}')

In [ ]:
# Format into Gemma multimodal chat template.
# IMPORTANT: store the image as a *path string*, not a PIL.Image.
# We then cast the column to datasets.Image() so PIL is loaded lazily at
# batch time. Avoids serialising 200k PIL objects into the .map disk cache,
# which is what blew the 20 GB Kaggle working quota last run.

from datasets import Image as ImageFeature

def to_chat(example):
    user_content = example['messages'][0]['content']
    image_rel = next(c['path'] for c in user_content if c['type'] == 'image')
    user_text = next(c['text'] for c in user_content if c['type'] == 'text')
    assistant_text = example['messages'][1]['content'][0]['text']
    return {
        'image': str(DATA_ROOT / image_rel),
        'conversations': [
            {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': user_text}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
        ],
    }

print('mapping train_ds → chat format...')
train_ds = train_ds.map(to_chat, remove_columns=train_ds.column_names, desc='train')
print('mapping val_ds   → chat format...')
val_ds = val_ds.map(to_chat, remove_columns=val_ds.column_names, desc='val')

# Cast 'image' column → lazy PIL.Image. The string path stays on disk;
# decoding happens only when a batch is fetched.
train_ds = train_ds.cast_column('image', ImageFeature())
val_ds = val_ds.cast_column('image', ImageFeature())

print(f'\n✓ ready: train={len(train_ds):,}  val={len(val_ds):,}')
print(f'  example image type: {type(train_ds[0]["image"]).__name__}  (lazy-loaded PIL)')
print(f'  example image size: {train_ds[0]["image"].size}')

## 3 — Load dataset

Expects the Kaggle dataset `pokou-ai-cocoa-training-data` mounted at `/kaggle/input/pokou-ai-cocoa-training-data/`.

In [ ]:
OUTPUT_DIR = f'/kaggle/working/cocoa_v1_{VARIANT}'

# T4 doesn't support bfloat16 — must use fp16 here.
# logging_steps=5 keeps the cell verbose so we can see training is alive.
# save_total_limit=1 + save_strategy='epoch' caps disk usage at one
# checkpoint (~1.5 GB for E2B). eval_strategy='no' skips per-step eval to
# protect the 12h kernel budget; we'll evaluate manually at the end.

PER_DEVICE_BATCH = 4 if VARIANT == 'e2b' else 2
GRAD_ACCUM = 2 if VARIANT == 'e2b' else 4
NUM_EPOCHS = 1   # one epoch on 5k samples ≈ 30 min on T4. Bump to 3 once
                 # you're happy with the smoke-test run.

print(f'training plan:')
print(f'  variant         = {VARIANT}')
print(f'  per-device bsz  = {PER_DEVICE_BATCH}')
print(f'  grad accum      = {GRAD_ACCUM}')
print(f'  effective bsz   = {PER_DEVICE_BATCH * GRAD_ACCUM}')
print(f'  epochs          = {NUM_EPOCHS}')
print(f'  steps/epoch     ≈ {len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM)}')
print(f'  output_dir      = {OUTPUT_DIR}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=5,            # verbose — every ~5 steps we see loss
        eval_strategy='no',          # skip eval to save disk + time
        save_strategy='epoch',
        save_total_limit=1,         # only keep 1 checkpoint
        bf16=False,                  # T4 ≠ bf16
        fp16=True,                   # T4 = fp16
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field='conversations',
        report_to='none',
        seed=42,
        dataloader_num_workers=2,
    ),
)
print('\n✓ trainer initialised')

In [ ]:
# Format into Gemma multimodal chat template
from PIL import Image

def to_chat(example):
    messages = example['messages']
    user_content = messages[0]['content']
    assistant_content = messages[1]['content']
    image_rel = next(c['path'] for c in user_content if c['type'] == 'image')
    image_path = str(DATA_ROOT / image_rel)
    user_text = next(c['text'] for c in user_content if c['type'] == 'text')
    assistant_text = assistant_content[0]['text']
    return {
        'image': Image.open(image_path).convert('RGB'),
        'conversations': [
            {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': user_text}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
        ],
    }

train_ds = train_ds.map(to_chat)
val_ds = val_ds.map(to_chat)

In [ ]:
import time

# Print GPU baseline so the resource panel + this log agree.
torch.cuda.empty_cache()
print(f'GPU mem before train: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print(f'starting training at {time.strftime("%H:%M:%S")}\n')

t0 = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - t0

print(f'\n✓ training finished in {elapsed / 60:.1f} min')
print(f'  final loss      = {trainer_stats.training_loss:.4f}')
print(f'  total steps     = {trainer_stats.global_step}')
print(f'  GPU mem peak    = {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

In [ ]:
OUTPUT_DIR = f'/kaggle/working/cocoa_v1_{VARIANT}'

# E2B runs the larger batch comfortably on a single T4; E4B needs smaller per-device batch
PER_DEVICE_BATCH = 4 if VARIANT == 'e2b' else 2
GRAD_ACCUM = 2 if VARIANT == 'e2b' else 4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='epoch',
        save_total_limit=3,
        bf16=True,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field='conversations',
        report_to='none',
        seed=42,
    ),
)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import subprocess
size = subprocess.check_output(['du', '-sh', OUTPUT_DIR]).decode().split()[0]
print(f'✓ LoRA adapter saved to {OUTPUT_DIR} ({size})')

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
GGUF_OUTPUT = f'/kaggle/working/cocoa_v1_{VARIANT}_gguf'

print(f'exporting GGUF Q4_K_M to {GGUF_OUTPUT} (this can take 5–10 min)...')
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method='q4_k_m')
print('✓ export done')

!ls -lh {GGUF_OUTPUT}/
import subprocess
size = subprocess.check_output(['du', '-sh', GGUF_OUTPUT]).decode().split()[0]
print(f'\n✓ total: {size}')

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('LoRA adapter saved to', OUTPUT_DIR)

## 7 — Export to GGUF Q4_K_M for on-device inference

Size targets:
- **E2B Q4_K_M**: ~1.3–1.6 GB file, runs in ~2 GB RAM on llama.cpp (fits 2–3 GB phones)
- **E4B Q4_K_M**: ~2.5–3.0 GB file, needs ~4 GB RAM (premium phones only)

In [ ]:
GGUF_OUTPUT = f'/kaggle/working/cocoa_v1_{VARIANT}_gguf'
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method='q4_k_m')
!ls -lh {GGUF_OUTPUT}/

## 8 — Quick sanity check (single-image inference)

In [ ]:
from transformers import TextStreamer

sample = val_ds[0]
inputs = tokenizer.apply_chat_template(
    sample['conversations'][:1],
    add_generation_prompt=True,
    return_tensors='pt',
    tokenize=True,
).to('cuda')

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, max_new_tokens=400, streamer=streamer, do_sample=False)

## 9 — Train the other variant

To produce both models for the submission, restart the kernel, set `VARIANT = 'e4b'` in cell 2, and re-run from Section 3. Outputs land in a separate folder (`cocoa_v1_e4b_gguf/`), so nothing is overwritten.